# NIHR Downloads Pipeline

The UK National Institute for Health and Care Research (NIHR) provides access to core data assets including funding information and allows for download of CSV files. Datasets are available at: https://fundingawards.nihr.ac.uk.

This notebook will:

1. Download two NIHR datasets. The NIHR Infrastructure Supported Projects Datasets and the NIHR Research and Training Awards Dataset
2. Dump the resulting files into the target DATA_INPUT_DIRECTORY
3. Read each file in the directory in
4. Write out a json-lines format file in the OIC datamodel to the designated data output directory. Output location MUST be a directory to allow for batching of the outputs into multiple files

In [1]:
from pathlib import Path
from zipfile import ZipFile
import json
from itertools import batched
from attrs import asdict
import polars as pl
from typing import List, Dict, Any
from datetime import datetime, date
import requests
from oic_scrape.items import AwardParticipant, AwardItem


In [10]:
DATA_INPUT_DIRECTORY = Path("../data/wellcome/input_data")
FUNDER_ORG_NAME = "National Institute for Health and Care Research (NIHR)"
FUNDER_ROR_ID = "https://ror.org/0187kwz082"
OUTPUT_LOCATION = Path("../data/wellcome")
OUTPUT_FORMAT = "jsonl"

In [4]:
# Download data
nihr_api_root = "https://nihr.opendatasoft.com/api/explore/v2.1/catalog/datasets/{identifier}/exports/json/"
datasets = ["nihr-summary-view"] #, "nihr-infrastructure-supported-projects"]
for dataset in datasets:
    r = requests.get(nihr_api_root.format(identifier=dataset))
    r.raise_for_status()
    with open(DATA_INPUT_DIRECTORY/ f"{dataset}.jsonl", "w") as f:
        json.dump(r.json(), f)



In [11]:
# Process NIHR Summary View
awards = []
json_file = DATA_INPUT_DIRECTORY / "nihr-summary-view.jsonl"
with open(json_file, "r") as f:
    j = json.load(f)
    for grant in j:
        award = AwardItem(
            _crawled_at=datetime.now(),
            source='https://nihr.opendatasoft.com/',
            grant_id=f"nihr::{grant.get('project_id')}",
            funder_org_name=grant.get('funder') if grant.get('funder') else FUNDER_ORG_NAME,
            funder_org_ror_id=FUNDER_ROR_ID,
            recipient_org_name=grant.get('contracted_organisation'),
            named_participants=[AwardParticipant(
                full_name=str(grant.get('award_holder_name')),
                grant_role=grant.get('involvement_type'),
                identifiers=dict(
                    orcid=grant.get('orcid')
                ),
                is_pi=True if grant.get('involvement_type') == 'Chief Investigator' else False
            )],
            grant_year=date.fromisoformat(grant.get('start_date')).year if grant.get('start_date') else None,
            grant_start_date=date.fromisoformat(grant.get('start_date')) if grant.get('start_date') else None,
            grant_end_date=date.fromisoformat(grant.get('end_date')) if grant.get('end_date') else None,
            award_amount=grant.get('award_amount_from_dh'),
            award_currency='GBP',
            grant_title=grant.get('project_title'),
            grant_description=grant.get('plain_english_abstracty'),
            program_of_funder=f'{grant.get('programme')}>{grant.get('programme_stream')}>{grant.get('funding_stream')}',
            raw_source_data = json.dumps(grant),
        )
        awards.append(asdict(award))


In [3]:
# Process NIHR Infrastructure Supported Projects
# No funding information is provided so skipping this currently

# json_file = DATA_INPUT_DIRECTORY / "nihr-infrastructure-supported_projects.jsonl"
# with open(json_file, "r") as f:
#     j = json.load(f)
#     for grant in j:
#         award = AwardItem(
#             _crawled_at=datetime.now(),
#             source='https://nihr.opendatasoft.com/',
#             grant_id=f"nihr::{grant.get('id')}",
#             funder_org_name=FUNDER_ORG_NAME,
#             funder_org_ror_id=FUNDER_ROR_ID,
#             recipient_org_name=grant.get('centre'),
#             named_participants=[AwardParticipant(
#                 full_name=grant.get('pi_full_name'),
#                 grant_role='principal_investigator',
#                 is_pi=True
#             )],
#             grant_year=date.fromisoformat(grant.get('start_date')).year if grant.get('start_date') else None,
#             grant_start_date=date.fromisoformat(grant.get('start_date')) if grant.get('start_date') else None,
#             grant_end_date=date.fromisoformat(grant.get('end_date')) if grant.get('end_date') else None,
#             award_amount=grant.get('award_amount_from_dh'),
#             award_currency='GBP',
#             grant_title=grant.get('project_title'),
#             grant_description=grant.get('plain_english_abstracty'),
#             program_of_funder=f'{grant.get('programme')}>{grant.get('programme_stream')}>{grant.get('funding_stream')}',
#             raw_source_data = json.dumps(grant),
#         )
#         awards.append(asdict(award))


In [12]:
pl.DataFrame(awards).write_ndjson(OUTPUT_LOCATION / f'nihr.jsonl')